# AbstractTensor — Manifold, tangent bundle and frame bundle

This notebook explains the structures defined with @def_manifold macro and their hierarchy 

---
## 1. Loading package

In [1]:
using AbstractTensors

[ Info: Precompiling AbstractTensors [55c8b40c-cbfa-4141-803e-4831c9841971] (cache misses: include_dependency fsize change (4), wrong source (2), mismatched flags (14))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


---
## 2. Manifolds and def_manifold macro
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of vector bundles with manifold as base manifold


In [3]:
?@def_manifold

```julia
@def_manifold name dim coord_indices basis_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frame bundles.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

Bind in the caller's scope:

  * `name`            → a [`Manifold`](@ref) instance
  * `tangent<name>`   → a [`VBundle`](@ref) instance (`isdual = false`)
  * `cotangent<name>` → a [`VBundle`](@ref) instance (`isdual = true`)
  * `frame<name>`, `coframe<name>`, moving basis symbols (default `e`, `θ`)

Coordinate indices (first list) → [`CoordinateIndex`](@ref):

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Basis indices (second list) → [`BasisIndex`](@ref):

```julia
A1          # BasisIndex(:A1, :tangentM)   — contravariant
-A1         # BasisIndex(:A1, :cotangentM) — covariant
```

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_manifold M d [b1, b2, b3, b4] [B1, B2, B3, B4]   # parametric dimension
```


In [4]:
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

Defined manifold M of dimension 4
Defined vector bundle tangentM with coordinate basis ∂ and its dual cotangentM with coordinate basis dx
Defined frame bundle frameM (moving frame e) and coframe bundle coframeM (moving coframe θ) over M


In [5]:
_MANIFOLDS

Dict{Symbol, Manifold} with 1 entry:
  :M => Manifold(M, dim=4, TBundle=tangentM, CBundle=cotangentM)

### b. (co)Tangent bundle and (co)Frame bundle 

In [6]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name      # :tangentM
tangentM.manifold  # :M
tangentM.dim       # 4
tangentM.isdual    # false
tangentM.dual      # :cotangentM
tangentM.coordinate_indices  # [CoordinateIndex(:a1, :tangentM), ...]
tangentM.basis_indices       # [BasisIndex(:A1, :tangentM), ...]
tangentM.bases     # [Basis(:∂, :tangentM, :coordinate), Basis(:e, :tangentM, :moving)]
```

### Fields

  * `name`     : bundle name, e.g. `:tangentM`
  * `manifold` : base manifold name, e.g. `:M`
  * `dim`      : fibre dimension
  * `isdual`   : false = contravariant (upper) slots, true = covariant (lower) slots.              Authoritative for index variance via [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`     : name of the paired dual bundle, e.g. `:cotangentM` or `:dualE`
  * `coordinate_indices` : [`CoordinateIndex`](@ref) for the coordinate chart (∂ / `dx`);              nonempty on tangent/cotangent bundles from [`@def_manifold`](@ref)
  * `basis_indices` : [`BasisIndex`](@ref) for fibre / moving bases (`e` / `θ`);              populated on tangent/cotangent by [`@def_manifold`](@ref) and on custom              bundles by [`@def_vbundle`](@ref)


In [7]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(∂, tangentM, coordinate), Basis(e, tangentM, moving)])

In [8]:
display(tangentM.bases)
display(tangentM.coordinate_indices)
display(tangentM.basis_indices)

2-element Vector{Basis}:
 Basis(∂, tangentM, coordinate)
 Basis(e, tangentM, moving)

4-element Vector{CoordinateIndex}:
 +a1 ∈ tangentM (coord)
 +a2 ∈ tangentM (coord)
 +a3 ∈ tangentM (coord)
 +a4 ∈ tangentM (coord)

4-element Vector{BasisIndex}:
 +A1 ∈ tangentM (basis)
 +A2 ∈ tangentM (basis)
 +A3 ∈ tangentM (basis)
 +A4 ∈ tangentM (basis)

In [9]:
cotangentM

VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4, bases=[Basis(dx, cotangentM, coordinate), Basis(θ, cotangentM, moving)])

In [10]:
display(cotangentM.bases)
display(cotangentM.coordinate_indices)
display(cotangentM.basis_indices)

2-element Vector{Basis}:
 Basis(dx, cotangentM, coordinate)
 Basis(θ, cotangentM, moving)

4-element Vector{CoordinateIndex}:
 -a1 ∈ cotangentM (coord)
 -a2 ∈ cotangentM (coord)
 -a3 ∈ cotangentM (coord)
 -a4 ∈ cotangentM (coord)

4-element Vector{BasisIndex}:
 -A1 ∈ cotangentM (basis)
 -A2 ∈ cotangentM (basis)
 -A3 ∈ cotangentM (basis)
 -A4 ∈ cotangentM (basis)

In [11]:
dx

Basis(dx, cotangentM, coordinate)

In [12]:
@doc Basis

```julia
Basis
```

A named frame for a vector bundle, of a given type (`:coordinate` or `:moving`). Instances are normally obtained from [`@def_manifold`](@ref) (coordinate and moving) or [`@def_frame_bundle`](@ref) (moving only on custom bundles), which bind variables such as `dx`, `∂`, `e`, and `θ` in the caller's scope.

### Fields

  * `name`    : display name, e.g. `:dx`, `:∂`, `:e`, `:θ`
  * `vbundle` : the bundle this frame is for, e.g. `:cotangentM`
  * `type`    : `:coordinate` (natural frame) or `:moving` (user-defined frame)

### Construction

The struct constructor takes three **symbols** (note the leading `:`):

```julia
Basis(:dx, :cotangentM, :coordinate)
Basis(:∂,  :tangentM,   :coordinate)
Basis(:e,  :tangentM,   :moving)
```

Lookup without the bound variable:

```julia
basis_for_vbundle(:cotangentM; type=:coordinate)  # same object as dx
bases_for_vbundle(:tangentM)                       # coordinate + moving bases
```

### Indexing

Indexing a `Basis` with an [`AbstractIndex`](@ref) from the **dual** bundle produces a [`BasisElement`](@ref):

```julia
dx[a1]     # a1 ∈ tangentM   → cotangentM coordinate element
e[-A1]     # -A1 ∈ cotangentM → tangentM moving element
```


In [13]:
display(cotangentM.bases)

2-element Vector{Basis}:
 Basis(dx, cotangentM, coordinate)
 Basis(θ, cotangentM, moving)

In [14]:
dx

Basis(dx, cotangentM, coordinate)

In [15]:
θ

Basis(θ, cotangentM, moving)

In [16]:
display(tangentM.bases)

2-element Vector{Basis}:
 Basis(∂, tangentM, coordinate)
 Basis(e, tangentM, moving)

In [17]:
∂

Basis(∂, tangentM, coordinate)

In [18]:
e

Basis(e, tangentM, moving)

In [19]:
@doc BasisElement

```julia
BasisElement
```

A single basis vector in a coordinate or moving frame for a (co)tangent vector bundle, labeled by an index from the dual bundle. Form it by applying a [`Basis`](@ref) to an index, e.g. `dx[a1]` or `e[-A1]`.

Generalize to a basis vector on any vector bundle (after [`@def_vbundle`](@ref)) by applying a [`Basis`](@ref) to an index from the dual vector bundle.

### Fields

  * `basis` : the [`Basis`](@ref) this element belongs to
  * `index` : the [`AbstractIndex`](@ref) labeling this element;           its vbundle is the **dual** of `basis.vbundle`

    dx[a1]   → BasisElement(Basis(:dx, :cotangentM, :coordinate), CoordinateIndex(:a1, :tangentM))   θ[A1]    → BasisElement(Basis(:θ,  :cotangentM, :moving),     BasisIndex(:A1, :tangentM))   ∂[-a1]   → BasisElement(Basis(:∂,  :tangentM,   :coordinate), CoordinateIndex(:a1, :cotangentM))   e[-A1]   → BasisElement(Basis(:e,  :tangentM,   :moving),     BasisIndex(:A1, :cotangentM))


In [20]:
dx[a1]

dx[a1]

In [21]:
θ[A1]

θ[A1]

In [22]:
∂[-a1]

∂[-a1]

In [23]:
e[-A1]

e[-A1]

#### Error handling 

In [35]:
dx[-a1]

LoadError: Cannot form dx[a1]: basis dx lives on :cotangentM (cotangent), so the label must be an upper (contravariant) index from :tangentM. Example: dx[a1]. Got index :a1 on :cotangentM (lower).

In [36]:
dx[A1]

LoadError: Cannot form dx[A1]: dx is a coordinate frame; use a coordinate index from :tangentM, e.g. dx[a1]. For the moving frame use a basis index, e.g. θ[A1]. Got basis index :A1.

#### Change of basis coordinate <-> moving

### c. Frame bundle 

In [31]:
frameM

VBundle,tangentM
Dual,coframeM
Moving basis,e


In [32]:
coframeM

VBundle,cotangentM
Dual,frameM
Moving basis,θ


---
## 3. Tensors
---

### a. Type and definition

In [33]:
@doc Tensor

```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold       # :M  (Symbol key into _MANIFOLDS)
T.slots          # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries     # [SlotSymmetry(n=2, order=2)]
T.is_traceless   # false
T.known_traces   # Any[]  (populated later)
T.print_as       # :T
T.metric         # :g or nothing
T.rank           # 2      (derived — length of slots)
```

### Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.                   Built from the index signs in the `@def_tensor` expression.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`. Use convenience                   constructors `symmetric`, `antisymmetric`, etc.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display name. Controls how the tensor appears in `show`                   and (later) LaTeX output.
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.


In [34]:
@doc @def_tensor

```julia
@def_tensor name[slot1, slot2, ...] manifold
@def_tensor name[slot1, slot2, ...] manifold  symmetries=[S1,...]  traceless=bool  print_as=:sym  metric=g
```

Define a new abstract tensor on `manifold` and bind it to `name` in the caller's scope.

### Slot syntax

  * `+idx (or idx)` : contravariant (upper) slot — index lives in `tangentM` or in the `vector bundle` specified in the input of `@def_vbundle` macro.
  * `-idx` : covariant (lower) slot — index lives in `cotangentM` or in the `dual vector bundle` specified by `@def_vbundle` macro.

The index symbols (`idx`) must already be registered to `tangentM` via `@def_manifold` or to a given vector bundle via `@def_vbundle` or latter with `@add_indices` macro. They are used only for validation; the tensor stores vbundle names per slot, not the defining symbols.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display name. Defaults to `name`. Example: `print_as=:R`.
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution:                 - one metric on manifold → assigned silently                 - multiple metrics → `@warn`, first defined is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_metric g[-a1, -a2] M

@def_tensor T[-a1, -a2] M                                   # metric auto-resolved to :g
@def_tensor F[-a1, -a2] M symmetries=[antisymmetric(2)]
@def_tensor R[-a1,-a2,-a3,-a4] M symmetries=[riemann_symmetry()]
@def_tensor W[-a1,-a2,-a3,-a4] M symmetries=[riemann_symmetry()] traceless=true print_as=:Weyl
@def_tensor mixed_T[a1, -a2] M   # mixed (1,1) tensor
```


In [37]:
@def_tensor T[-a1, -a2] M

┌ Warning: No metric is defined on manifold M. Tensor T has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/AbstractTensors/src/tensors.jl:358


Defined tensor T on manifold M with 2 slot(s), metric=nothing


In [39]:
@doc basis_expansion

```julia
basis_expansion(te::TensorExpression; frame::Symbol=:coordinate) -> BasisExpansion
```

Expand `te` into its formal basis representation using the `:coordinate` (default) or `:moving` frame.

For each slot, the corresponding [`BasisElement`](@ref) is constructed from the registered frame of the given `type`, with the index flipped to the dual vbundle:

```julia
basis_expansion(T[-a1,-a2])                  # → T[-a1,-a2] dx[a1] ⊗ dx[a2]
basis_expansion(T[-a1,-a2]; frame=:moving)   # → T[-a1,-a2] θ[A1]  ⊗ θ[A2]
basis_expansion(T[a1, a2])                   # → T[a1,a2]   ∂[-a1] ⊗ ∂[-a2]
basis_expansion(T[a1,-a2])                   # → T[a1,-a2]  ∂[-a1] ⊗ dx[a2]
```

Errors if no frame of the requested type has been registered for any slot's bundle.

```julia
basis_expansion(T::Tensor; frame::Symbol=:coordinate) -> BasisExpansion
```

Expand `T` using canonical slot indices (the first `rank(T)` indices from the manifold's tangent bundle, each assigned the slot's vbundle). Delegates to `basis_expansion(::TensorExpression; frame=frame)`.


In [40]:
basis_expansion(T)  

T[-a1, -a2] dx[a1] ⊗ dx[a2]

In [41]:
T[-a1,-a2]

T[-a1, -a2]

In [43]:
T(∂[-a1],∂[-a2])

LoadError: MethodError: objects of type Tensor are not callable
The object of type `Tensor` exists, but no method is defined for this combination of argument types when trying to treat it as a callable object.